# Annotation Merge Notebook

This notebook merges the manually annotated dataset with the original cleaned dataset to restore metadata columns.

Output:
data/annotated_data/final_annotated_dataset.csv

This dataset is used for model training.

In [7]:
from pathlib import Path
import pandas as pd

# Go from notebooks → project root
PROJECT_ROOT = Path.cwd().parent

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\LENOVO\OneDrive\Documents\Strath\Masters\sentiment-analysis-tool


In [12]:
# Load original and annotated datasets
ORIGINAL_PATH = PROJECT_ROOT / "data" / "annotated_data" / "annotation_file.xlsx" 

ANNOTATED_PATH = PROJECT_ROOT / "data" / "annotated_data" / "manually_annotated_data.xlsx"

df_original = pd.read_excel(ORIGINAL_PATH)
df_annotated = pd.read_excel(ANNOTATED_PATH)

print("Original:", df_original.shape)
print("Annotated:", df_annotated.shape)

Original: (52923, 11)
Annotated: (2000, 5)


In [22]:
#Install rapidfuzz for fuzzy matching
import sys
!{sys.executable} -m pip install rapidfuzz

     ---------------------------------------- 1.5/1.5 MB 12.3 kB/s eta 0:00:00



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
#Test

from rapidfuzz import process, fuzz

print("RapidFuzz installed successfully")

RapidFuzz installed successfully


In [24]:
# Clean text for matching

def normalize(text):
    text = str(text).lower()
    text = text.strip()
    return text

df_original["match_text"] = df_original["text"].apply(normalize)
df_annotated["match_text"] = df_annotated["text"].apply(normalize)

In [25]:
# Perform fuzzy matching

matches = []

original_texts = df_original["match_text"].tolist()

for text in df_annotated["match_text"]:

    match = process.extractOne(
        text,
        original_texts,
        scorer=fuzz.token_sort_ratio
    )

    matched_text = match[0]
    score = match[1]

    row = df_original[df_original["match_text"] == matched_text].iloc[0]

    matches.append({
        "source": row["source"],
        "date": row["date"],
        "channel": row["channel"],
        "product": row["product"],
        "language": row["language"],
        "clean_text": row["clean_text"],
        "match_score": score
    })

df_matches = pd.DataFrame(matches)

In [26]:
# Combine annotated data with matched original data

df_final = pd.concat(
    [df_annotated.reset_index(drop=True),
     df_matches.reset_index(drop=True)],
    axis=1
)

In [27]:
# Analyze match scores

print(df_final["match_score"].describe())

count    2000.000000
mean       97.289181
std         4.245855
min        62.422635
25%        96.078431
50%        99.086758
75%       100.000000
max       100.000000
Name: match_score, dtype: float64


In [28]:
#

df_final[df_final["match_score"] < 85]

,text,topic,sentiment,annotator,confidence,match_key,match_text,source,date,channel,product,language,clean_text,match_score
41,all im requesting a reversal of sh that i sent...,Payments,negative,annotator_1,high,all im requesting a reversal of sh that i sent...,all im requesting a reversal of sh that i sent...,CRM Tool,01/12/2025 00:00,Service Email,MPESA Transactions,english,all requesting a reversal of sh i sent to a re...,84.125145
139,equity im sending this email with more concern...,Payments,negative,annotator_1,high,equity im sending this email with more concern...,equity im sending this email with more concern...,CRM Tool,05/11/2025 00:00,Service Email,Western Union,english,equity sending this email with more concerned ...,74.301819
171,hello iwant to open the app but is not working,Mobile Banking App,negative,annotator_1,high,hello iwant to open the app but is not working,hello iwant to open the app but is not working,CRM Tool,04/11/2025 00:00,Service Email,Mobile,english,iwant to open the app but is not working,83.333333
419,my app aint opening,Mobile Banking App,negative,annotator_1,high,my app aint opening,my app aint opening,CRM Tool,01/11/2025 00:00,Service Email,Mobile,unknown,app aint opening,82.608696
484,name assist with attached ticket been followin...,Payments,negative,annotator_1,high,name assist with attached ticket been followin...,name assist with attached ticket been followin...,CRM Tool,17/12/2025 00:00,Service Web,Float Management,unknown,name assist with attached ticket been followin...,79.879880
626,name why s my equity app not working,Mobile Banking App,negative,annotator_1,high,name why s my equity app not working,name why s my equity app not working,CRM Tool,09/11/2025 00:00,Service Email,Mobile,unknown,why s my equity app not working,81.818182
771,you are well i went to the branch in office an...,Customer Service,negative,annotator_1,high,you are well i went to the branch in office an...,you are well i went to the branch in office an...,CRM Tool,03/11/2025 00:00,Service Email,Card Transaction Disputes,english,you are well i went to the branch in office an...,81.950207
784,help me make my app while here in,Mobile Banking App,neutral,annotator_1,high,help me make my app while here in,help me make my app while here in,CRM Tool,04/11/2025 00:00,Service Email,Mobile,unknown,namename help me make my app while here in nam...,70.967742
814,accountnumber number accountnumber amp for na...,Acccount Issues,neutral,annotator_1,high,accountnumber number accountnumber amp for na...,accountnumber number accountnumber amp for na...,CRM Tool,01/12/2025 00:00,Service Web,MPESA Transactions,unknown,to reverse erroneous transaction from accountn...,77.934272
820,advise client to avail vehicle for valuation a...,Insurance Services,neutral,annotator_1,high,advise client to avail vehicle for valuation a...,advise client to avail vehicle for valuation a...,CRM Tool,02/12/2025 00:00,Service Web,Motor Vehicle,unknown,advise client to avail vehicle for valuation a...,78.846154


In [29]:

# Save final annotated dataset

OUTPUT_PATH = PROJECT_ROOT / "data" / "annotated_data" / "final_annotated_dataset.xlsx"

df_final.to_excel(OUTPUT_PATH, index=False)

print("Saved:", OUTPUT_PATH)

Saved: c:\Users\LENOVO\OneDrive\Documents\Strath\Masters\sentiment-analysis-tool\data\annotated_data\final_annotated_dataset.xlsx
